# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore record sets, fields, and column IDs of the dataset
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets detected in metadata. The dataset package may expose its records via single Table record set.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- {rs['@id']}: {rs.get('name', '')}")

# We'll now programmatically list record set @id, its fields' @id and their dataTypes
print("\n---- Record sets and their fields: ----")
for rs in dataset.record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}")
    field_specs = rs.get('field', [])
    if isinstance(field_specs, dict):
        field_specs = [field_specs]
    for f in field_specs:
        print(f"  Field '@id': {f['@id']} | name: {f.get('name', '')} | dataType: {f.get('dataType', '')}")
    if not field_specs:
        print("  (No field definitions found)")

if not record_sets:
    print("\n`mlcroissant` will default to exposing all tabular records as a single record set.
To browse the content, use:")
    # List some records by using the default record set id, which is the dataset url
    for i, rec in enumerate(dataset.records()):
        print(rec)
        if i > 1:
            break

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Croissant references entities by their `@id`. We'll load the main record set for the data table.

In [ ]:
# Determine the main record set @id (default is the croissant url if single tabular set)
main_record_set_id = croissant_url  # This is standard when only one set is present
# If you found a specific @id above, you may set it directly:
# main_record_set_id = '<record_set_@id>'

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"DataFrame columns for record set {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping. Croissant fields are referenced by their `@id`.

In [ ]:
# List some numeric fields for EDA (based on field names and manual inspection)
numeric_candidates = [col for col in df.columns if 'Age' in col or 'Interval' in col or 'Years' in col or 'age' in col or 'interval' in col or df[col].dtype.kind in 'fi']
print("Numeric field candidates for EDA:", numeric_candidates)

# Choose a numeric field by `@id` (auto-detected or manual, here assumed as 'Age_at_2nd_CRC')
numeric_field_id = 'Age_at_2nd_CRC'
if numeric_field_id not in df.columns:
    # fallback to any numeric field found
    import numpy as np
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

threshold = 60
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (showing up to 5):")
print(filtered_df.head())

# Normalize numeric field
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Assume a group field (using 'Sex' as example, referenced by @id if possible)
group_field_id = 'Sex'
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the normalized age distribution after filtering
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=10, kde=True)
plt.title(f'Normalized {numeric_field_id} Distribution (>{threshold})')
plt.xlabel(f'Normalized {numeric_field_id}')
plt.ylabel('Count')
plt.show()

# Visualize numerical field vs. group field
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the `mlcroissant` library.

- We loaded metadata and records directly from the Croissant schema URL.
- We inspected the record set, its fields, and their `@id`s.
- We extracted the main data table and performed EDA: filtering by age at 2nd CRC, normalizing, and grouping by sex.
- We visualized the distribution of age at 2nd CRC and its stratification by sex.

This workflow can be extended for more advanced analysis, including modeling, advanced stratification, or integrating additional biomedical knowledge sources.